# NSE intraday risk controls — Group 17838

Run the current M4 pilot entirely in **Google Colab**: setup → Drive → analysis → tables → save.
Use a **CPU runtime**. The analysis runs on Google's computer, not your laptop.

This reproduces the frozen **January–March 2021 pilot: five stocks and the NSE index**. It does not run the final 2021–2023 holdout or add unfinished funded-account/risk analysis. The tested research functions remain in the repository; this notebook is the single place to run them.

Make a copy of this notebook in Drive. Add shortcuts to the instructor's **NSE Cash Stocks Data** and **NSE Index Data** folders under **My Drive**; no data upload is needed. Keep the two paths in step 3 consistent with your shortcut names. Each group member needs their own data access.


## 1. Set up and check the code
Only the archive reader needs installing. The research code uses Python's standard library. This notebook uses the fixed `colab-v1` release; the executed commit is recorded with your results.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tempfile

REPO = Path('/content/nse-intraday-risk-controls')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'colab-v1',
                    'https://github.com/I-am-Uchenna/nse-intraday-risk-controls.git', str(REPO)], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
PINNED = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'colab-v1^{commit}'], text=True).strip()
if COMMIT != PINNED or subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Use a fresh Colab runtime: the existing code differs from the fixed release.')
if not shutil.which('bsdtar'):
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'libarchive-tools'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-q'], cwd=REPO, env={**os.environ, 'PYTHONPATH': str(REPO / 'src')}, check=True)
print('Code version:', COMMIT)


## 2. Connect your Google Drive
Approve Google's Drive connection when prompted. Source archives are read only. Selected compressed files and the working database are staged on Colab's temporary disk; results are saved to your Drive in step 5. Nothing is downloaded to your laptop.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Run the frozen pilot
The cash folder must contain `Cash Data January 2021.rar`, `Cash Data February 2021.rar` and `Cash Data March 2021.rar`; the index folder must contain `NSE-Index-2021.7z`. Original outer ZIP downloads are also accepted. Do not supply duplicate copies of the same source.

The loader streams one CSV at a time into a small SQLite file, and the replay reads one session at a time. Each run gets a fresh temporary folder. Only change the two folder paths; keep the frozen strategy settings unchanged.


In [ ]:
CASH = Path('/content/drive/MyDrive/NSE Cash Stocks Data/2021')
INDEX = Path('/content/drive/MyDrive/NSE Index Data')
for folder in (CASH, INDEX):
    if not folder.is_dir():
        raise FileNotFoundError(f'Check your Drive shortcut and folder path: {folder}')

WORK = Path(tempfile.mkdtemp(prefix='nse-m4-', dir='/content'))
OUTPUT = WORK / 'results'
CONFIG = REPO / 'config/m4_pilot.json'
# ponytail: reuse the checked engine; keep only the notebook workflow here.
subprocess.run([sys.executable, 'scripts/prepare_pilot.py', '--downloads', str(CASH),
                '--downloads', str(INDEX), '--work-dir', str(WORK), '--config', str(CONFIG)], cwd=REPO, check=True)
subprocess.run([sys.executable, 'scripts/run_pilot.py', '--work-dir', str(WORK),
                '--config', str(CONFIG), '--output-dir', str(OUTPUT)], cwd=REPO, check=True)
print('Completed:', OUTPUT)


## 4. Read the results
These are conditional development results. Basis points here mean daily P&L divided by fixed reference capital, not a continuously funded wealth return. Both candle orderings and both timestamp assumptions are shown. The chart uses the reference cost/delay and stop-first path only; the full CSVs retain every declared setting.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

summary = json.loads((OUTPUT / 'summary.json').read_text())
means = pd.read_csv(OUTPUT / 'scenario_means.csv')
contrasts = pd.read_csv(OUTPUT / 'paired_contrasts.csv')
print(summary['evidence_label'])
print('Files:', summary['input_files'], '| Rows:', summary['input_rows'])
display(means[means['setting'].eq('reference')].round(3))
display(contrasts[contrasts['setting'].eq('reference')].round(3))
plot_data = means[means['setting'].eq('reference') & means['path'].eq('O-L-H-C')]
ax = plot_data.pivot(index='policy', columns='timestamp_assumption', values='mean_net_daily_return_bp').plot.bar(rot=0, figsize=(8, 4))
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Q1 2021 development pilot: reference settings, stop-first path',
       xlabel='Exit policy', ylabel='Mean daily P&L / reference capital (bp)')
ax.legend(title='Timestamp assumption')
plt.tight_layout()
plt.savefig(OUTPUT / 'reference_comparison.png', dpi=160)
plt.show()


## 5. Save results to Drive
Save before the runtime disconnects. This keeps the aggregate tables, chart, frozen settings, input audit and code version in a new run folder. The source archives stay in their original folders. The temporary price database and detailed ledgers are not copied or published.

The final holdout and funded-account work remain separate research tasks. A successful rerun confirms reproducibility of this pilot, not executable profitability.


In [ ]:
SAVE = Path('/content/drive/MyDrive/WQU_Capstone_17838/runs') / WORK.name
SAVE.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT, SAVE)  # Refuse to overwrite a saved run.
shutil.copy2(CONFIG, SAVE / 'm4_pilot.json')
shutil.copy2(WORK / 'input_audit.json', SAVE / 'input_audit.json')
(SAVE / 'code_version.txt').write_text(COMMIT + '\n')
print('Saved to Google Drive:', SAVE)


**Practical notes.** Colab sessions are temporary and resources vary. The browser still uses some laptop memory, but the market computation and working data use the hosted runtime. If a session ends, run the notebook again; no source file is overwritten. Repository access will need updating when the group makes the repository private. See [Google's Colab FAQ](https://research.google.com/colaboratory/faq.html) for Drive I/O and runtime limits.
